In [1]:
import pandas as pd
from IPython.display import display

# ============================================================
# 0. Load final dataset with distress labels (SCALED)
# ============================================================

path_scaled_labels = "../data/final/firm_macro_with_distress_scaled.csv"
df = pd.read_csv(path_scaled_labels)

print("Full scaled dataset with labels. Shape:", df.shape)
display(df.head())

# ============================================================
# 1. Basic checks & type casting
# ============================================================

required_cols = [
    "gvkey", "fyear",
    "event_year", "years_to_event",
    "post_event_or_event_year",
    "distress_1y", "distress_2y", "distress_3y",
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["gvkey"] = df["gvkey"].astype(str)
df["fyear"] = pd.to_numeric(df["fyear"], errors="coerce").astype("Int64")

print("\nDistress 1y label distribution:")
print(df["distress_1y"].value_counts(dropna=False))

# ============================================================
# 2. Choose horizon and define target variable
# ============================================================

# Change this value to 2 or 3 if you veux un autre horizon:
HORIZON = 1

target_col = f"distress_{HORIZON}y"
print(f"\nUsing '{target_col}' as target variable (label).")

if target_col not in df.columns:
    raise ValueError(f"{target_col} not found in dataframe.")

# ============================================================
# 3. Filter to pre-event / non-post-event observations
#    (on enlève les années >= année de faillite)
# ============================================================

df_model = df[df["post_event_or_event_year"] == 0].copy()
print("\nShape after removing event-year and post-event years:", df_model.shape)

print("\nTarget distribution in modeling sample:")
print(df_model[target_col].value_counts(dropna=False))

# ============================================================
# 4. Define feature set X and metadata
# ============================================================

# ID / meta columns qu'on garde pour info mais PAS dans X
id_meta_cols = [
    "gvkey", "fyear", "datadate",
    "conm", "tic", "sic", "fyr",
    "dlrsn", "bankruptcy_dlrsn",
    "event_year", "years_to_event",
    "post_event_or_event_year",
    "distress_1y", "distress_2y", "distress_3y",
]

id_meta_cols = [c for c in id_meta_cols if c in df_model.columns]

# Features = toutes les colonnes moins ID/meta
feature_cols = [c for c in df_model.columns if c not in id_meta_cols]

print("\nNumber of feature columns:", len(feature_cols))
print("Example feature cols:", feature_cols[:10])

# Construction de X et y
X = df_model[feature_cols].copy()
y = df_model[target_col].astype(int)

# Pour garder aussi l'identité / temps avec X+y
df_model_final = df_model[id_meta_cols].copy()
df_model_final["target"] = y
# on ajoute aussi toutes les features dans le même fichier
for c in feature_cols:
    df_model_final[c] = X[c]

print("\nFinal modeling dataframe shape (with IDs + features + target):",
      df_model_final.shape)
display(df_model_final.head())

# ============================================================
# 5. Save modeling-ready dataset to data/final/
# ============================================================

output_path = f"../data/final/modeling_dataset_{HORIZON}y_scaled.csv"
df_model_final.to_csv(output_path, index=False)

print(f"\nModeling dataset saved to: {output_path}")
print("You can now load this file in the modeling notebook for train/test split + models.")


Full scaled dataset with labels. Shape: (242802, 35)


,gvkey,datadate,fyear,conm,tic,dlrsn,bankruptcy_dlrsn,roa,ebitda_margin,debt_ratio,...,us_cpi_-_all_urban:_all_items_sadj,us_treasury_bill_rate_-_3_month_(ep)_nadj,us_treasury_yield_adjusted_to_constant_maturity_-_20_year_nadj,us_unemployment_rate_sadj,event_year,years_to_event,post_event_or_event_year,distress_1y,distress_2y,distress_3y
0,1004,1995-05-31,1994,AAR CORP,AIR,NaN,0,0.226063,0.171086,-0.181664,...,-1.499747,1.514148,1.871440,0.317004,NaN,NaN,0,0,0,0
1,1004,1996-05-31,1995,AAR CORP,AIR,NaN,0,0.238559,0.171952,-0.183780,...,-1.434948,1.229085,1.641494,0.014970,NaN,NaN,0,0,0,0
2,1004,1997-05-31,1996,AAR CORP,AIR,NaN,0,0.246015,0.173069,-0.203332,...,-1.333876,1.261269,1.553465,-0.105844,NaN,NaN,0,0,0,0
3,1004,1998-05-31,1997,AAR CORP,AIR,NaN,0,0.255895,0.173950,-0.174871,...,-1.251054,1.353225,1.465435,-0.407879,NaN,NaN,0,0,0,0
4,1004,1999-05-31,1998,AAR CORP,AIR,NaN,0,0.260078,0.174173,-0.174890,...,-1.194974,0.948620,0.861805,-0.649506,NaN,NaN,0,0,0,0



Distress 1y label distribution:
distress_1y
0    242802
Name: count, dtype: int64

Using 'distress_1y' as target variable (label).

Shape after removing event-year and post-event years: (236776, 35)

Target distribution in modeling sample:
distress_1y
0    236776
Name: count, dtype: int64

Number of feature columns: 22
Example feature cols: ['roa', 'ebitda_margin', 'debt_ratio', 'total_debt_to_equity', 'current_ratio', 'cash_ratio', 'interest_coverage', 'cfo_margin', 'free_cash_flow', 'asset_turnover']

Final modeling dataframe shape (with IDs + features + target): (236776, 36)


,gvkey,fyear,datadate,conm,tic,dlrsn,bankruptcy_dlrsn,event_year,years_to_event,post_event_or_event_year,...,book_to_market,price_to_book,working_capital_to_assets,retained_earnings_to_assets,name,us_gdp_(ar)_cura,us_cpi_-_all_urban:_all_items_sadj,us_treasury_bill_rate_-_3_month_(ep)_nadj,us_treasury_yield_adjusted_to_constant_maturity_-_20_year_nadj,us_unemployment_rate_sadj
0,1004,1994,1995-05-31,AAR CORP,AIR,NaN,0,NaN,NaN,0,...,-0.003440,-0.148203,0.372367,0.214576,-1.624936,-1.398798,-1.499747,1.514148,1.871440,0.317004
1,1004,1995,1996-05-31,AAR CORP,AIR,NaN,0,NaN,NaN,0,...,-0.003440,-0.087432,0.376548,0.215062,-1.551124,-1.357335,-1.434948,1.229085,1.641494,0.014970
2,1004,1996,1997-05-31,AAR CORP,AIR,NaN,0,NaN,NaN,0,...,-0.003440,-0.042469,0.377995,0.214125,-1.444150,-1.283458,-1.333876,1.261269,1.553465,-0.105844
3,1004,1997,1998-05-31,AAR CORP,AIR,NaN,0,NaN,NaN,0,...,-0.003440,-0.000877,0.309188,0.213540,-1.337176,-1.197468,-1.251054,1.353225,1.465435,-0.407879
4,1004,1998,1999-05-31,AAR CORP,AIR,NaN,0,NaN,NaN,0,...,0.138547,-0.096157,0.300008,0.214827,-1.230202,-1.114746,-1.194974,0.948620,0.861805,-0.649506



Modeling dataset saved to: ../data/final/modeling_dataset_1y_scaled.csv
You can now load this file in the modeling notebook for train/test split + models.
